# Cuaderno 6 — Agrupar y resumir

**Descripción y Visualización de Datos — UAI 2026 — Clase 6**

La clase 4 contaste cuánta gente menciona la delincuencia como el principal
problema del país. Salieron estos números:

| Año | Menciones |
|---|---|
| 1994 | 284 |
| 2025 | **1.342** |
| 2026 | 461 |

Leído así, la delincuencia se triplicó en 2025 y se desplomó al año siguiente.
**Es falso, y el error no está en R: está en la pregunta.** En 2025 la CEP
encuestó a 4.217 personas y en 2026 a 1.466. Estás comparando 1.342 de 4.217
contra 461 de 1.466, como si fueran lo mismo.

Contar sirve para describir un grupo. **Para comparar dos grupos hay que
dividir**, y eso es lo de hoy:

| | Qué hace |
|---|---|
| `summarise()` | convierte **muchas filas en una sola**: un promedio, un conteo, un porcentaje |
| `group_by()` | le dice a `summarise()` que haga eso **una vez por grupo** |
| `arrange()` | ordena la tabla que salió |

Con esas tres líneas, las noventa y seis mil respuestas de la CEP caben en una
tabla que se lee en diez segundos.

## Parte 0 — Punto de partida

Nada nuevo. La base grande demora unos segundos en bajar: son 19 MB.

In [ ]:
library(dplyr)

cep <- read.csv("https://raw.githubusercontent.com/naimbro/naimbro.github.io/main/materiales/2026_descripcion_visualizacion_datos/datos/cep_consolidada_1994_2026.csv")

nrow(cep)

In [ ]:
names(cep)

## Parte 1 — Calentamiento

Dos cadenas con lo de las clases pasadas. Si alguna no te sale sola, ésa es la
que hay que repasar hoy.

**1.** Quédate solo con el año 2026 y cuenta `problema_1` de mayor a menor.

*(`cep`, luego `filter()`, luego `count()` con `sort = TRUE`)*

In [ ]:
# Tu código acá

**2.** Ahora las mujeres de la Región Metropolitana en 2026. Son dos condiciones
a la vez, así que necesitas `&` dentro de un `filter()`.

In [ ]:
# Tu código acá

## Parte 2 — `summarise()` con siete filas

Antes de soltarte sobre noventa y seis mil respuestas, vamos a usar una base
diminuta: los siete experimentos sobre inteligencia artificial y desempeño que
miramos en la clase 4. Son **siete filas**, así que todo lo que devuelva R lo
puedes verificar contando con el dedo.

In [ ]:
exp <- read.csv("https://raw.githubusercontent.com/naimbro/naimbro.github.io/main/materiales/2026_descripcion_visualizacion_datos/datos/experimentos_ia_desempeno.csv")

exp %>% select(estudio, ambito, efecto_promedio_pct)

`summarise()` toma una tabla completa y devuelve **una sola fila**. Adentro se
escribe `nombre = cálculo`, igual que en `mutate()`, pero el cálculo aplasta la
columna entera en un número:

In [ ]:
exp %>%
  summarise(estudios = n(),
            efecto_promedio = mean(efecto_promedio_pct))

Siete estudios, y el promedio salió **`NA`**.

No es un error de R: es R siendo honesto. La fila de Física universitaria tiene
`NA` en `efecto_promedio_pct` porque ese estudio reportó su efecto en otra escala.
Y el promedio de algo que incluye un «no sé» es, correctamente, «no sé».

Para pedirle que lo ignore hay que decírselo explícitamente:

In [ ]:
exp %>%
  summarise(estudios = n(),
            efecto_promedio = mean(efecto_promedio_pct, na.rm = TRUE))

**5,83.** Cuéntalo a mano si quieres: 18 + 14 + 40 + 0 − 17 − 20, dividido en 6.

Fíjate en la trampa que acabas de armar: la tabla dice `estudios = 7`, pero el
promedio se calculó con **6**. `n()` cuenta filas, no cuenta datos válidos. Si
publicas «promedio de 7 estudios», estás mintiendo por una fila.

Se arregla pidiendo las dos cosas por separado:

In [ ]:
exp %>%
  summarise(estudios = n(),
            con_dato = sum(!is.na(efecto_promedio_pct)),
            efecto_promedio = mean(efecto_promedio_pct, na.rm = TRUE))

> `is.na(x)` pregunta «¿esto es `NA`?» y el `!` da vuelta la respuesta. Sumar
> `TRUE`s es contarlos: `sum(!is.na(x))` son **los casos que sí tienen dato**.

## Parte 3 — `group_by()`: la misma cuenta, una vez por grupo

Un solo promedio para los siete estudios esconde lo importante, porque cuatro
miden productividad en el trabajo y tres miden aprendizaje. `group_by()` va
justo **antes** de `summarise()` y parte la tabla en pedazos:

In [ ]:
exp %>%
  group_by(ambito) %>%
  summarise(estudios = n(),
            con_dato = sum(!is.na(efecto_promedio_pct)),
            efecto_promedio = mean(efecto_promedio_pct, na.rm = TRUE))

Ahí está la historia completa, y no se parece en nada al 5,83 de recién:

- **Trabajo: +18,0%** en promedio, con los cuatro estudios apuntando para arriba.
- **Aprendizaje: −18,5%**, y calculado con sólo dos de los tres estudios.

Un promedio general habría dicho «la IA sube un poco el desempeño». Agrupar dice
otra cosa: **sube la productividad de quien ya sabe y baja el aprendizaje de quien
está aprendiendo.** Misma base, misma función, una línea de diferencia.

**3.** Agrupa por `con_barandas` en vez de `ambito`. ¿Qué pasa con los estudios de
aprendizaje que sí pusieron barandas?

In [ ]:
# Tu código acá

## Parte 4 — Ahora sí: noventa y seis mil personas

Volvemos a la CEP y a la pregunta del principio. Primero, un número para toda la
base:

In [ ]:
cep %>% summarise(personas = n())

Ahora el porcentaje. Aquí hay un truco que vale la clase entera, y usa el
`ifelse()` de la semana pasada.

Para calcular «qué porcentaje menciona la delincuencia» se hace en dos pasos:
primero `mutate()` crea una columna que vale **1** cuando la respuesta es
delincuencia y **0** cuando no; después el **promedio de esa columna de unos y
ceros es exactamente la proporción**. Multiplicada por 100, el porcentaje.

In [ ]:
cep %>%
  mutate(delinc = ifelse(problema_1 == "Delincuencia, asaltos y robos", 1, 0)) %>%
  summarise(personas = n(),
            pct_delincuencia = mean(delinc) * 100)

**22,4% de todos los chilenos encuestados en treinta y dos años.** Un solo número
para tres décadas: cierto, y completamente inútil. Lo que el editor quiere es la
comparación. Agreguemos la línea que faltaba:

In [ ]:
cep %>%
  mutate(delinc = ifelse(problema_1 == "Delincuencia, asaltos y robos", 1, 0)) %>%
  group_by(anio) %>%
  summarise(personas = n(),
            pct_delincuencia = mean(delinc) * 100)

Treinta y dos filas: la serie completa de Chile. Dos cosas que mirar antes de
seguir.

Primero, **R sólo te muestra las diez primeras** y abajo avisa `22 more rows`.
Nota además que la tabla ahora dice `# A tibble` en vez de salir como las de
`count()`: `group_by()` devuelve un formato distinto, que imprime más ordenado y
menos filas.

Segundo, ahí está la respuesta a la pregunta del principio: la primera fila dice
**1994, 19,0%**. Para ver el otro extremo hay que ordenar. `count()` tenía su
`sort = TRUE`, pero `summarise()` no trae nada parecido, así que se ordena aparte
con `arrange()`, y `desc()` lo hace de mayor a menor:

In [ ]:
cep %>%
  mutate(delinc = ifelse(problema_1 == "Delincuencia, asaltos y robos", 1, 0)) %>%
  group_by(anio) %>%
  summarise(personas = n(),
            pct_delincuencia = mean(delinc) * 100) %>%
  arrange(desc(pct_delincuencia))

**2025 con 31,8% y 2026 con 31,4%.** No se triplicó ni se desplomó: subió doce
puntos en treinta años y ahí se quedó. La misma base, la misma columna, y una
conclusión completamente distinta a la que dieron los conteos.

**4.** La misma receta, pero agrupando por `zona` (urbano/rural) en vez de `anio`.
¿Le teme más a la delincuencia el campo o la ciudad?

In [ ]:
# Tu código acá

**5.** Ahora por `gse`, el nivel socioeconómico, y ordenado de mayor a menor.
Mira la columna `personas` antes de sacar conclusiones.

In [ ]:
# Tu código acá

**6.** Cambia de tema: en vez de la delincuencia, calcula el porcentaje que
**aprueba al presidente** (`aprueba_presidente == "Aprueba"`) por año, y ordénalo.
¿Cuál fue el mejor año y cuál el peor?

In [ ]:
# Tu código acá

## Parte 5 — Los tres controles antes de resumir

`summarise()` nunca te va a avisar que la tabla que produjo no significa nada.
Estos tres controles son los que hay que correr **antes** de publicar cualquier
comparación.

### Control 1 — ¿Están todos los grupos?

In [ ]:
cep %>% count(anio)

Mira la lista con calma. Después de 2019 viene **2021**.

**No hay encuesta 2020**: la CEP no salió a terreno en pandemia. Nada en tu tabla
de recién te lo dijo, y si dibujas una línea uniendo 2019 con 2021 vas a mostrar
una tendencia continua a través de un año que no existe.

### Control 2 — ¿La columna estuvo siempre?

`posicion_politica` parece una columna estupenda para comparar. Preguntémosle
cuántas respuestas válidas tiene cada año:

In [ ]:
cep %>%
  group_by(anio) %>%
  summarise(personas = n(),
            con_posicion = sum(posicion_politica != "")) %>%
  arrange(desc(anio))

**Desde 2021 la columna está completamente vacía.** Dejó de preguntarse, o dejó
de armonizarse en esta base consolidada. Da lo mismo la razón: cualquier análisis
político con los datos recientes es imposible, y hay que decirlo, no rodearlo.

Lo que sí se puede hacer es acotar el período a donde la columna existe:

In [ ]:
cep %>%
  filter(anio >= 2015 & anio <= 2019) %>%
  filter(posicion_politica == "Derecha" | posicion_politica == "Izquierda") %>%
  mutate(delinc = ifelse(problema_1 == "Delincuencia, asaltos y robos", 1, 0)) %>%
  group_by(posicion_politica) %>%
  summarise(personas = n(),
            pct_delincuencia = mean(delinc) * 100)

32,5% contra 16,7%: la brecha más grande que vas a encontrar en toda esta base.
Pero es una frase sobre 2015–2019, no sobre hoy, y así hay que escribirla.

### Control 3 — ¿Cuántos son en cada grupo?

El más importante de los tres, y el que se olvida siempre. Mira qué pasa si
pides el nivel socioeconómico sólo del último año:

In [ ]:
cep %>%
  filter(anio == 2026) %>%
  mutate(delinc = ifelse(problema_1 == "Delincuencia, asaltos y robos", 1, 0)) %>%
  group_by(gse) %>%
  summarise(personas = n(),
            pct_delincuencia = mean(delinc) * 100)

El segmento **E tiene 9 personas**. Nueve. Su porcentaje se ve igual de
respetable que el de C3, que tiene 791, y las dos cifras están en la misma tabla,
con la misma cantidad de decimales, alineadas con la misma prolijidad.

> **La regla del día:** antes de comparar dos grupos, pregúntale a cada uno
> cuántos son. **Un promedio sin `n()` es una opinión con decimales.**

**7.** Repite la tabla anterior, pero con toda la serie en vez de sólo 2026.
¿Cuántas personas tiene ahora el segmento E, y cambia la conclusión?

In [ ]:
# Tu código acá

## Parte 6 — Tu encargo

Acá abajo va el trabajo del bloque: el encargo que le tocó a tu grupo en la
redacción. Tienes que salir de esta celda con **dos cosas**:

1. Una **tabla** hecha con `group_by()` + `summarise()`, que incluya `n()`.
2. Un **titular de una frase**, con un número y una comparación adentro.

Y con la respuesta lista para las tres preguntas del editor: cuántos casos hay
detrás, sobre qué total está calculado el porcentaje, y qué le falta a la serie.

In [ ]:
# Encargo del grupo — la tabla

In [ ]:
# Encargo del grupo — el chequeo: n() por grupo, años faltantes, casos vacíos

In [ ]:
# Espacio libre para lo que se te ocurra probar

## Antes de irte

**Archivo → Guardar** (Ctrl+S).

### Lo que aprendiste hoy

| Para qué | Cómo se escribe |
|---|---|
| Muchas filas en una sola | `cep %>% summarise(personas = n())` |
| Un promedio | `summarise(edad_media = mean(edad))` |
| Ignorar los `NA` | `mean(edad, na.rm = TRUE)` |
| Contar los datos válidos | `sum(!is.na(edad))` |
| La misma cuenta por grupo | `group_by(anio) %>% summarise(...)` |
| Un porcentaje | `mutate(x = ifelse(cond, 1, 0))` y luego `mean(x) * 100` |
| Ordenar el resultado | `arrange(desc(pct))` |

### Las tres ideas

1. **Contar describe; dividir compara.** Si los grupos tienen distinto tamaño,
   los conteos absolutos mienten sin avisar.
2. **`group_by()` no hace nada solo.** Es una instrucción para el `summarise()`
   que viene después: hazlo una vez por grupo.
3. **Siempre pide `n()`.** Un grupo de nueve personas y uno de cuarenta mil se
   ven idénticos en una tabla, y sólo uno de los dos se puede publicar.